In [0]:
# Databricks notebook source
dbutils.widgets.removeAll()

# COMMAND ----------
# DBTITLE 1,Install libraries
%pip install recordlinkage -q

# COMMAND ----------
import logging
from datetime import datetime
import os

# Create logs directory if it doesn't exist
base_dir = os.getcwd()
log_dir = os.path.abspath(os.path.join(base_dir, "../../logs"))
os.makedirs(log_dir, exist_ok=True)

log_filename = f"{log_dir}/memberpersonbridge_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(log_filename.replace('file:', '')),
        logging.StreamHandler()
    ]
)

logger = logging.getLogger(__name__)
logger.info("FHIR Member Person Bridge Processing Started")
logger.info(f"Log file: {log_filename}")

# COMMAND ----------
# DBTITLE 1,Configuration
BRONZE_VOLUME_PATH = "/Volumes/claimspan/bronze/member_consolidated"
SILVER_BRIDGE_TABLE = "claimspan.silver.silver_memberpersonbridge"

logger.info("Configuration loaded:")
logger.info(f"  Source Volume Path: {BRONZE_VOLUME_PATH}")
logger.info(f"  Target Silver Table: {SILVER_BRIDGE_TABLE}")

# COMMAND ----------
from pyspark.sql.functions import (
    date_format, sha2, col, row_number, substring, 
    upper, trim, regexp_replace, expr, to_json, struct, concat_ws
)
from pyspark.sql.window import Window
import pandas as pd
import recordlinkage
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, LongType

# DBTITLE 1,Load and transform source data
def LoadSource(volumePath):
    logger.info(f"Starting LoadSource from volume: {volumePath}")
    df_bronze = spark.read.format("delta").load(volumePath)
    logger.info(f"Raw FHIR records loaded: {df_bronze.count()}")

    windowPartition = Window.partitionBy(col("FileID")).orderBy(col("FileID"))

    # Generate RecordHash using to_json(struct("*")) and build UniqueRecord using FileID-RowNumber composite key
    sparkMem_df = df_bronze.withColumn("RecordHash", sha2(to_json(struct("*")), 256)) \
        .distinct() \
        .withColumn("RowNumber", row_number().over(windowPartition)) \
        .withColumn("UniqueRecord", concat_ws("-", col("FileID"), col("RowNumber"))) \
        .withColumn("FileID", col("FileID")) \
        .withColumn("FileLayoutID", col("FileLayoutID")) \
        .withColumn("name_family", upper(trim(col("name_family")))) \
        .withColumn("name_given_first", upper(trim(col("name_given_first")))) \
        .withColumn("name_family_initial", upper(substring(trim(col("name_family")), 1, 1))) \
        .withColumn("name_given_first_initial", upper(substring(trim(col("name_given_first")), 1, 1))) \
        .withColumn("birthDate", expr("try_cast(birthDate as string)")) \
        .withColumn("birthDate_formatted", expr("regexp_replace(try_cast(birthDate as string), '[^0-9]', '')")) \
        .withColumn("gender", upper(trim(col("gender")))) \
        .withColumn("address_permanent_line1", upper(trim(col("address_permanent_line1")))) \
        .withColumn("telecom_phone_home", trim(col("telecom_phone_home"))) \
        .withColumn("telecom_phone_home_formatted", regexp_replace(trim(col("telecom_phone_home")), "[^0-9]", "")) \
        .withColumn("identifier_planMemberID", upper(trim(col("identifier_planMemberID")))) \
        .withColumn("identifier_beneficiaryID", upper(trim(col("identifier_beneficiaryID")))) \
        .withColumn("identifier_uniquepersonkey", upper(trim(col("identifier_uniquepersonkey")))) \
        .select(
            "UniqueRecord", "FileLayoutID", "FileID", "RowNumber", "name_family", "name_given_first", 
            "gender", "telecom_phone_home", "address_permanent_line1", "birthDate", "birthDate_formatted", 
            "identifier_planMemberID", "identifier_beneficiaryID", "identifier_uniquepersonkey", "name_family_initial", "name_given_first_initial", 
            "telecom_phone_home_formatted"
        )

    logger.info(f"Processed FHIR records after transformations: {sparkMem_df.count()}")
    logger.info("LoadSource completed successfully")
    return sparkMem_df

# COMMAND ----------
# DBTITLE 1,Define matching rules
RuleMBIColumns = ["identifier_beneficiaryID", "name_given_first_initial", "name_family_initial", "birthDate_formatted", "telecom_phone_home_formatted", "address_permanent_line1", 11]
RulePMIDColumns = ["identifier_planMemberID", "name_given_first_initial", "name_family_initial", "birthDate_formatted", "telecom_phone_home_formatted", "address_permanent_line1", 11]
RuleUPKColumns = ["identifier_uniquepersonkey", "name_given_first_initial", "name_family_initial", "birthDate_formatted", "telecom_phone_home_formatted", "address_permanent_line1", 11]
RuleOtherColumns = ["birthDate_formatted", "name_given_first", "name_family", "telecom_phone_home_formatted", "address_permanent_line1", 14]

RulesAll = [RuleMBIColumns, RulePMIDColumns, RuleUPKColumns, RuleOtherColumns]
CompareColumns = [
    "name_family_initial", "name_given_first_initial", "name_family", "name_given_first", "identifier_beneficiaryID", 
    "identifier_planMemberID", "identifier_uniquepersonkey", "birthDate_formatted", "telecom_phone_home_formatted", "address_permanent_line1"
]

# COMMAND ----------
# DBTITLE 1,Record linkage - Compare function
def RulesToCompare(rules, pandasMem_df):
    logger.info("Starting RulesToCompare function")
    matchesAllRules_df = pd.DataFrame()
    for i, lst in enumerate(rules, 1):
        indexer = recordlinkage.Index()
        indexer.block(lst[0])
        candidatesBlock = indexer.index(pandasMem_df)
        logger.info(f"Rule {i} ({lst[0]}): Found {len(candidatesBlock)} candidate pairs")
        
        compareBlock = recordlinkage.Compare()
        threshold = lst[-1]
        for col_name in CompareColumns:
            compareBlock.exact(col_name, col_name, label=str(col_name))
            
        features = compareBlock.compute(candidatesBlock, pandasMem_df)
        features[lst[0]] = features[lst[0]].apply(lambda x: x * 10)
        
        matchesRule = features[features[lst[:-1]].sum(axis=1) >= threshold]
        matchesRule_df = matchesRule.index.to_frame()
        logger.info(f"Rule {i}: {len(matchesRule)} matches found (threshold: {threshold})")
        
        if len(matchesRule_df) > 0:
            matchesAllRules_df = pd.concat([matchesAllRules_df, matchesRule_df])
            
    logger.info(f"RulesToCompare completed. Total matches across all rules: {len(matchesAllRules_df)}")
    return matchesAllRules_df

# COMMAND ----------
# DBTITLE 1,Record linkage - Run execution logic
def RunLinking(sparkMem_df):
    logger.info("=" * 50)
    logger.info("STEP 1: Entering RunLinking function")
    logger.info("=" * 50)
    
    pandasMem_df = sparkMem_df.toPandas()
    
    # Handle duplicate UniqueRecord values by creating a truly unique index
    if pandasMem_df["UniqueRecord"].duplicated().any():
        logger.warning("UniqueRecord column contains duplicates - creating unique index")
        duplicate_count = pandasMem_df["UniqueRecord"].duplicated().sum()
        logger.warning(f"Found {duplicate_count} duplicate UniqueRecord values")
        
        # Create a truly unique index by combining UniqueRecord with row position
        pandasMem_df = pandasMem_df.reset_index(drop=True)
        pandasMem_df["UniqueRecord"] = pandasMem_df.index.astype(str) + "_" + pandasMem_df["UniqueRecord"].astype(str)
    
    pandasMem_df = pandasMem_df.set_index("UniqueRecord", drop=False)
    
    logger.info("STEP 2: Evaluating Record Linkage rules via RulesToCompare...")
    dfCombined = RulesToCompare(RulesAll, pandasMem_df)
    
    # Helper to force all pandas columns into string and build explicit Spark StructType string schema
    def create_string_spark_df(pdf):
        pdf_str = pdf.astype(str)
        schema = StructType([StructField(c, StringType(), True) for c in pdf_str.columns])
        return spark.createDataFrame(pdf_str, schema=schema)

    if dfCombined.empty:
        logger.info("NO MATCHES FOUND across any rules. Falling back to self-assignment path.")
        finalPandas_df = pandasMem_df.copy()
        finalPandas_df['MatchID'] = finalPandas_df['UniqueRecord']
        return create_string_spark_df(finalPandas_df)
    
    logger.info("STEP 3: Cleaning MultiIndex column structural headers")
    dfCombined.columns = [0, 1]
    
    logger.info("STEP 4: Reshaping pairs for directional alignment (A->B and B->A)")
    dfMatchedColumnA = dfCombined.rename(columns={0: "A", 1: "B"})
    dfMatchedColumnB = dfCombined.rename(columns={0: "B", 1: "A"})
    
    dfMatched = pd.concat([dfMatchedColumnA, dfMatchedColumnB], ignore_index=True)
    dfMatched.drop_duplicates(inplace=True)
    
    logger.info("STEP 5: Formatting cross-reference master table (Record <-> Match)")
    matchesAll_df = pd.concat([
        dfMatched[["A", "B"]].rename(columns={"A": "Record", "B": "Match"}),
        dfMatched[["B", "A"]].rename(columns={"B": "Record", "A": "Match"})
    ]).reset_index(drop=True)
    
    matchesSame_df = matchesAll_df[["Record", "Record"]].copy()
    matchesSame_df.columns = ['Record', 'Match']
    
    matchesAll_df = pd.concat([matchesAll_df, matchesSame_df], ignore_index=True)
    matchesAll_df.drop_duplicates(inplace=True)
    
    logger.info("STEP 6: Generating single anchor map grouping via Column 'A'")
    distinctRow = dfMatched.groupby('A').head(1).drop('B', axis=1)
    
    matchesAll_df = pd.merge(matchesAll_df, distinctRow, how="left", left_on="Record", right_on="A").drop('A', axis=1)
    
    logger.info("STEP 7: Performing lookups against parent frame using Index references")
    matchesAll_df = pd.merge(matchesAll_df, pandasMem_df["UniqueRecord"], left_on="Record", right_index=True).rename(columns={"UniqueRecord": "RecordID"})
    matchesAll_df = pd.merge(matchesAll_df, pandasMem_df["UniqueRecord"], left_on="Match", right_index=True).rename(columns={"UniqueRecord": "MatchID"})
    
    logger.info("STEP 8: Aggregating structural match arrays per RecordID")
    matched_df = matchesAll_df.groupby("RecordID") \
            .agg({"MatchID": lambda x: list(pd.unique(x))}) \
            .reset_index()
    matched_df["MatchID"] = matched_df["MatchID"].apply(lambda x: sorted(x))
    
    logger.info("STEP 9: Stripping layout metadata metrics to create clean joining payload")
    matchesAllModified = matchesAll_df.groupby("RecordID").head(1)
    drop_cols = ['MatchID', 'Match', 'Record']
    existing_drops = [c for c in drop_cols if c in matchesAllModified.columns]
    matchesAllModified = matchesAllModified.drop(columns=existing_drops)
    if 'index' in matchesAllModified.columns:
        matchesAllModified = matchesAllModified.drop(columns=['index'])
        
    newDFToMatch = pd.merge(matched_df, matchesAllModified, how="left", on="RecordID")
    
    logger.info("STEP 10: Merging linkage clusters back to master source pandas frame")
    finalPandas_df = pandasMem_df.reset_index(drop=True).merge(newDFToMatch, left_on="UniqueRecord", right_on="RecordID", how="left")
    
    finalPandas_df['MatchID'] = finalPandas_df['MatchID'].fillna(finalPandas_df['UniqueRecord'])
    
    logger.info("RunLinking COMPLETED SUCCESSFULLY.")
    return create_string_spark_df(finalPandas_df)

# COMMAND ----------
# DBTITLE 1,Define SQL transformations
finalSQL = """
WITH ESAIPersonWithIdentifiers AS (
  SELECT 
     UniqueRecord, FileLayoutID, FileID, RowNumber, name_family, name_given_first, birthDate, gender, 
     address_permanent_line1, telecom_phone_home, identifier_planMemberID, identifier_beneficiaryID, identifier_uniquepersonkey,
     CASE WHEN INSTR(MatchID, ',') = 0 THEN MatchID ELSE SUBSTR(MatchID, 2, INSTR(MatchID, ',') - 2) END AS MatchID,
     ROW_NUMBER() OVER(PARTITION BY (CASE WHEN INSTR(MatchID, ',') = 0 THEN MatchID ELSE SUBSTR(MatchID, 2, INSTR(MatchID, ',') - 2) END) ORDER BY FileID ASC, RowNumber ASC) AS FirstPersonIdentifier,
     ROW_NUMBER() OVER(PARTITION BY (CASE WHEN INSTR(MatchID, ',') = 0 THEN MatchID ELSE SUBSTR(MatchID, 2, INSTR(MatchID, ',') - 2) END) ORDER BY FileID DESC, RowNumber DESC) AS CurrentPersonIdentifier
  FROM ESAICompletePersonTable
),
MemberPersonBridge AS (
  SELECT 
     fp.UniqueRecord AS ESAIInternalPersonID,
     CASE WHEN cp.CurrentPersonIdentifier = 1 THEN 1 ELSE 0 END AS IsCurrent,
     cp.UniqueRecord, cp.FileLayoutID, cp.FileID, cp.RowNumber, cp.name_family, cp.name_given_first, cp.birthDate, 
     cp.gender, cp.address_permanent_line1, cp.telecom_phone_home, cp.identifier_planMemberID, cp.identifier_beneficiaryID, 
     cp.identifier_uniquepersonkey, cp.MatchID
  FROM ESAIPersonWithIdentifiers cp
  LEFT JOIN ESAIPersonWithIdentifiers fp 
    ON cp.MatchID = fp.MatchID AND fp.FirstPersonIdentifier = 1
),
MemberPersonBridge_CurrPlanMbr AS (
  SELECT 
     ESAIInternalPersonID, IsCurrent, UniqueRecord, FileLayoutID, FileID, RowNumber, name_family, name_given_first, 
     birthDate, gender, address_permanent_line1, telecom_phone_home, identifier_planMemberID, identifier_beneficiaryID,
     IFNULL(NULLIF(identifier_planMemberID, 'None'), '') AS PlanMemberIdModified,
     IFNULL(NULLIF(identifier_uniquepersonkey, 'None'), '') AS UniquePersonKeyModified,
     identifier_uniquepersonkey, MatchID,
     CASE WHEN IFNULL(identifier_planMemberID, 'None') = 'None' THEN NULL 
          WHEN ROW_NUMBER() OVER(PARTITION BY identifier_planMemberID ORDER BY COALESCE(FileID, 0) DESC, COALESCE(RowNumber, 0) DESC) = 1 THEN 1 
          ELSE 0 END AS IsCurrentPlanMemberID,
     CASE WHEN IFNULL(identifier_uniquepersonkey, 'None') = 'None' THEN NULL 
          WHEN ROW_NUMBER() OVER(PARTITION BY identifier_uniquepersonkey ORDER BY COALESCE(FileID, 0) DESC, COALESCE(RowNumber, 0) DESC) = 1 THEN 1 
          ELSE 0 END AS IsCurrentUniquePersonKey,
     CASE WHEN ESAIInternalPersonID = UniqueRecord THEN 1 ELSE 0 END AS IsOriginalMemberID 
  FROM MemberPersonBridge
),
PUModPop AS (
  SELECT *,
      CASE WHEN PlanMemberIdModified <> '' THEN 1 ELSE 0 END AS IsPlanMemberIdPopulated,
      CASE WHEN UniquePersonKeyModified <> '' THEN 1 ELSE 0 END AS IsUniquePersonKeyModifiedPopulated,
      CONCAT(PlanMemberIdModified, '-', UniquePersonKeyModified) AS PMUP
  FROM MemberPersonBridge_CurrPlanMbr
),
Final AS (
  SELECT *,
      CAST(COALESCE(IsCurrentPlanMemberID, IsCurrentUniquePersonKey) AS STRING) AS IsCurrentPMUP
  FROM PUModPop
)
SELECT 
   ESAIInternalPersonID,
   IsCurrent,
   UniqueRecord,
   CAST(FileLayoutID AS INT) AS FileLayoutID,
   FileID,
   name_family,
   name_given_first,
   birthDate,
   gender,
   address_permanent_line1,
   telecom_phone_home,
   identifier_planMemberID,
   identifier_beneficiaryID,
   identifier_uniquepersonkey,
   sha2(concat_ws('|',
     IFNULL(ESAIInternalPersonID,''), IFNULL(CAST(IsCurrent AS STRING),''), IFNULL(UniqueRecord,''),
     IFNULL(CAST(FileLayoutID AS STRING),''), IFNULL(CAST(FileID AS STRING),''), IFNULL(name_family,''),
     IFNULL(name_given_first,''), IFNULL(birthDate,''), IFNULL(gender,''), IFNULL(address_permanent_line1,''),
     IFNULL(telecom_phone_home,''), IFNULL(identifier_planMemberID,''), IFNULL(identifier_beneficiaryID,''), IFNULL(identifier_uniquepersonkey,''),
     IFNULL(CAST(IsCurrentPlanMemberID AS STRING),''), IFNULL(CAST(IsCurrentUniquePersonKey AS STRING),''),
     IFNULL(CAST(IsOriginalMemberID AS STRING),''), IFNULL(PMUP,''), IFNULL(TRY_CAST(IsCurrentPMUP AS STRING),'')
   ), 256) AS hashKey,
   IsCurrentPlanMemberID,
   IsCurrentUniquePersonKey,
   IsOriginalMemberID,
   PMUP,
   TRY_CAST(IsCurrentPMUP AS INT) AS IsCurrentPMUP
FROM Final
"""

# COMMAND ----------
# DBTITLE 1,Main execution - Process and write to Silver
logger.info("=" * 60)
logger.info("MAIN EXECUTION STARTED FOR FHIR")
logger.info("=" * 60)

sparkMem_df = LoadSource(BRONZE_VOLUME_PATH)
numRows = sparkMem_df.count()
logger.info(f"Total FHIR records to process: {numRows}")

if numRows == 0:
    print("No records to process")
    logger.info("No records to process - exiting")
elif numRows == 1:
    print("Single record - skipping linkage, processing directly")
    logger.info("Single record detected - skipping linkage")
    convertedSpark_df = sparkMem_df.withColumn("MatchID", col("UniqueRecord"))
    convertedSpark_df = convertedSpark_df.withColumn("FileID", col("FileID").cast(LongType())).withColumn("RowNumber", col("RowNumber").cast(LongType()))
    
    convertedSpark_df.createOrReplaceTempView("ESAICompletePersonTable")
    temp_df = spark.sql(finalSQL)
    
    if temp_df.filter("IsCurrentPMUP IS NULL").count() > 0:
        logger.error("Validation failed: records contain both PlanMemberID and UniquePersonKey")
        raise Exception("Failed as records contain both a PlanMemberID and UniquePersonKey")
    else:
        temp_df.write.format("delta").mode("append").saveAsTable(SILVER_BRIDGE_TABLE)
        logger.info("Silver FHIR layer processing completed successfully (single record path)")
else:
    print("Running record linkage...")
    logger.info("Multiple records detected - running record linkage")
    try:
        convertedSpark_df = RunLinking(sparkMem_df)
    except Exception as e:
        logger.error(f"RunLinking failed: {str(e)}")
        raise
        
    convertedSpark_df = convertedSpark_df.withColumn("FileID", col("FileID").cast(LongType())).withColumn("RowNumber", col("RowNumber").cast(LongType()))
    convertedSpark_df.createOrReplaceTempView("ESAICompletePersonTable")
    
    temp_df = spark.sql(finalSQL)
    
    if temp_df.filter("IsCurrentPMUP IS NULL").count() > 0:
        logger.error("Validation failed: records contain both PlanMemberID and UniquePersonKey")
        raise Exception("Failed as records contain both a PlanMemberID and UniquePersonKey")
    else:
        temp_df.write.format("delta").mode("append").saveAsTable(SILVER_BRIDGE_TABLE)
        print("\nSilver layer processing for FHIR completed successfully!")
        logger.info("Silver layer processing for FHIR completed successfully")

In [0]:
member = spark.read.format("parquet").load("/Volumes/claimspan/bronze/member")
display(member)

In [0]:
member_consolidated = spark.read.format("delta").load("/Volumes/claimspan/bronze/member_consolidated")
display(member_consolidated)